# Whole-frame image-to-video with LTX-2.5 - birthday test set

**Author**: Konrad Jelen (kj) <br>
**Date**: 2026-10-25 <br>
**Model**: LTX-2.5 22B dev checkpoint, bf16, `Lightricks/LTX-2.5@49b093a39c19169a8a81721dcc541e07e945232f` <br>
**Photo set**: `henryk-urodziny-2026`, three working copies <br>

Renders one 10-second live-photo clip per birthday photo with LTX-2.5, a whole-frame image-to-video model that redraws the photo and animates it from a text prompt. Every photo is rendered once from the Star Colonel's prompt, timed and measured, so he can view the clips and judge whether they look believable and natural; the notebook measures and does not rank.

## Approach

1. **Pin the GPU by UUID** - the ComfyUI server that renders and the FILM interpolation run on the RTX PRO 6000; this kernel makes no render.
2. **Load the working copy** - each PNG at its own resolution; the graph scales and centre-crops it to the model's 3:4 size.
3. **Prompt each photo** - the Star Colonel's generic sentence verbatim, then a part written from that photo's own moment.
4. **Render through the module** - `henryk_media.animation.image_to_video` starts the server, fills the graph markers, submits, times the render, samples peak VRAM and stops the server.
5. **Convert to 30 fps** - the graph renders 241 frames at 24 fps; FILM interpolation, not duplicated frames, delivers 300 frames at 30 fps.
6. **Save and check** - every clip gets a sidecar and a checkpoint; the cells below list what was written and show each clip under its photo.

## Outputs

- `tmp/media/ltx25-i2v-probe/<stem>.mp4` - three clips, 300 frames at 30 fps, 768 x 1024
- `<stem>.md` beside each clip - source, model and weight files, prompt, negative prompt, seed, size, steps, cfg, GPU, seconds, peak VRAM, interpolation
- `checkpoints/<stem>.json` - the same record, read back on a re-run so a finished clip is not rendered again


## GPU Selection

Pins the RTX PRO 6000 Blackwell (nvidia-smi index 1) by UUID before any import that reaches torch. The ComfyUI server and the FILM worker receive the same UUID from the setup file; this kernel only reads the device name for the Configuration render.


In [ ]:
import os

# Deterministic ordering so a UUID always names the same physical card
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# RTX PRO 6000 Blackwell, pinned by UUID from `nvidia-smi -L`; the server gets the same pin
os.environ["CUDA_VISIBLE_DEVICES"] = "GPU-a44e514a-aa5b-2844-35f1-4be28e1921e5"


## Imports

All imports in one cell. The project package comes after the GPU pin because `henryk_media.animation.common` imports torch. The notebook calls the module; the module talks to the server over HTTP.


In [ ]:
%load_ext autoreload
%autoreload 2

# Standard library
import os  # Paths relative to the notebook for image and video links
from pathlib import Path  # Clip file names in the tables

# Notebook display
from IPython.display import Image, Markdown, Video, display  # Photos and clips shown by path, never embedded

# Rich console output
from rich import box  # Table border style
from rich import print as rprint  # Formatted console printing
from rich.console import Group  # A table and a paragraph in one output block
from rich.progress import (  # Progress bar components
    BarColumn,
    MofNCompleteColumn,
    Progress,
    SpinnerColumn,
    TextColumn,
    TimeElapsedColumn,
)
from rich.table import Table  # Tabular output

# PyTorch - device details for the Configuration render; the module imports it as well
import torch

# Project package
from henryk_media.animation import common  # Clip paths, sidecars, checkpoints, working copies
from henryk_media.animation import image_to_video as i2v  # Server, graph fill, render, interpolation
from henryk_media.config import INTERIM_DATA_DIR, PROJ_ROOT  # Project paths


## Reproducibility

Seeds Python, NumPy and PyTorch in this kernel. The render's own randomness is fixed by the seed passed to the graph, so the same seed, prompt and settings redraw the same clip.


In [ ]:
common.set_seed(42)


## Configuration

Paths, the model's native size and rate, the fixed seed, the sampler settings and the negative prompt. Every value below is written into the sidecar.

- **Size** - 768 x 1024, the closest LTX-2.5 size to the photos' 3:4 portrait aspect, in multiples of 32 (0.79 MP, one stage)
- **Length** - 241 frames, the 8k+1 grid at the native 24 fps, which is 10.04 s; the module delivers 300 frames at 30 fps
- **Sampler** - 30 steps, Euler, the dev checkpoint's full (SFT) settings; the graph carries the `LTXVScheduler` flow-matching defaults
- **CFG** - dual guidance, video 3.0 and audio 7.0; the graph generates a sound track, which the module drops when it encodes the 30 fps clip
- **Guidance** - spatio-temporal guidance 1.0 on block 28, modality guidance 3.0
- **First frame** - image strength 1.0, so the clip starts from the photo's own pixels


In [ ]:
# Inputs
PHOTO_SET = "henryk-urodziny-2026"  # folder name under data/interim
PHOTO_NAMES = [  # the three working copies, in render order
    "IMG_8840",  # Henry holding a glass jar up to the camera, a girl at the right edge
    "IMG_8564",  # Henry posing with a picture book among his classmates
    "IMG_8732",  # Henry pausing over a drawing with a blue pencil
]
WORKING_COPIES = [INTERIM_DATA_DIR / PHOTO_SET / f"{name}.png" for name in PHOTO_NAMES]

# Output
OUTPUT_DIR = PROJ_ROOT / "tmp/media/ltx25-i2v-probe"  # clips, sidecars and checkpoints/

# The model's server, graph and native rate, recorded by the setup stage
SETUP_FILE = PROJ_ROOT / "tmp/workflows/image-to-video/setup-chosen.json"
SETUP = i2v.Setup.load(SETUP_FILE)

# Render
WIDTH, HEIGHT = 768, 1024  # closest LTX-2.5 size to the 3:4 portrait aspect, multiples of 32
LENGTH = 241  # 8k+1 frames at the native 24 fps = 10.04 s
SEED = 42  # fixed, so a re-render with the same prompt redraws the same clip

# Model settings recorded in the sidecar
MODEL_SETTINGS = {
    "weights": (
        "ltx-2.5-22b-dev-transformer-bf16, gemma4-12b-with-proj-ltx-2.5-bf16, "
        "ltx-2.5-video-vae-bf16, ltx-2.5-audio-vae-bf16"
    ),
    "steps": 30,  # dev checkpoint, full settings
    "sampler": "euler",
    "scheduler": "LTXVScheduler max_shift 2.05 base_shift 0.95 stretch terminal 0.1",
    "cfg_video": 3.0,  # dual CFG, video
    "cfg_audio": 7.0,  # dual CFG, audio
    "stg_scale": 1.0,  # spatio-temporal guidance
    "stg_blocks": "28",
    "modality_scale": 3.0,
    "image_strength": 1.0,  # the first frame is the photo itself
}

# Negative prompt: the LTX-2.5 reference default, verbatim, plus six terms against camera motion and redraw artefacts
OFFICIAL_NEGATIVE_PROMPT = (
    "blurry, out of focus, overexposed, underexposed, low contrast, washed out colors, excessive noise, "
    "grainy texture, poor lighting, flickering, motion blur, distorted proportions, unnatural skin tones, "
    "deformed facial features, asymmetrical face, missing facial features, extra limbs, disfigured hands, "
    "wrong hand count, artifacts around text, inconsistent perspective, camera shake, incorrect depth of "
    "field, background too sharp, background clutter, distracting reflections, harsh shadows, inconsistent "
    "lighting direction, color banding, cartoonish rendering, 3D CGI look, unrealistic materials, uncanny "
    "valley effect, incorrect ethnicity, wrong gender, exaggerated expressions, wrong gaze direction, "
    "mismatched lip sync, silent or muted audio, distorted voice, robotic voice, echo, background noise, "
    "off-sync audio, incorrect dialogue, added dialogue, repetitive speech, jittery movement, awkward "
    "pauses, incorrect timing, unnatural transitions, inconsistent framing, tilted camera, flat lighting, "
    "inconsistent tone, cinematic oversaturation, stylized filters, or AI artifacts."
)
NEGATIVE_PROMPT = (
    f"{OFFICIAL_NEGATIVE_PROMPT} camera movement, zoom, changing face, morphing, distorted hands, "
    "extra fingers, text"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"
gpu_compute_cap = (
    f"{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}"
    if torch.cuda.is_available()
    else "N/A"
)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

rprint(
    f"""[bold medium_purple]Configuration[/bold medium_purple]
[dim]{"─" * 44}[/dim]
[bold slate_blue1]Model[/bold slate_blue1]
  Name: [cyan]{SETUP.model}[/cyan]
  Revision: [cadet_blue]{SETUP.revision}[/cadet_blue]
  Port: [grey70]{SETUP.port}[/grey70]
  Native rate: [light_sea_green]{SETUP.native_fps:g} fps[/light_sea_green]

[bold slate_blue1]Render[/bold slate_blue1]
  Size: [light_sea_green]{WIDTH} x {HEIGHT} px[/light_sea_green]
  Length: [light_sea_green]{LENGTH} frames at {SETUP.native_fps:g} fps = {LENGTH / SETUP.native_fps:.2f} s[/light_sea_green]
  Delivery: [light_sea_green]{i2v.FRAMES} frames at {i2v.FPS} fps = {i2v.FRAMES / i2v.FPS:.2f} s[/light_sea_green]
  Seed: [dark_sea_green]{SEED}[/dark_sea_green]
  Steps: [grey70]{MODEL_SETTINGS['steps']}[/grey70]
  CFG: [grey70]video {MODEL_SETTINGS['cfg_video']:g}, audio {MODEL_SETTINGS['cfg_audio']:g}[/grey70]

[bold slate_blue1]Paths[/bold slate_blue1]
  Photos: [dim]{INTERIM_DATA_DIR / PHOTO_SET}[/dim]
  Clips: [dim]{OUTPUT_DIR}[/dim]

[bold]Device[/bold]
  Using: [green]{device}[/green]
  GPU: [cyan]{gpu_name}[/cyan]
  Compute cap: [yellow]{gpu_compute_cap}[/yellow]
  Memory: [yellow]{gpu_memory_gb:.1f} GB[/yellow]"""
)


## Data Loading

Reads each working copy for its size and colour profile. The graph scales and centre-crops the photo to 768 x 1024, so the clip size is fixed; the three photos are within 0.4 % of the 3:4 aspect, so the crop removes almost nothing.


In [ ]:
photo_table = Table(title="Working copies", box=box.SIMPLE_HEAVY)
for column in ("Photo", "Width x height", "Aspect", "Colour profile"):
    photo_table.add_column(
        column,
        style="grey70" if column == "Photo" else None,
        justify="right" if column == "Width x height" else "left",
    )
for photo in WORKING_COPIES:
    height, width = common.load_working_copy(photo).shape[:2]
    photo_table.add_row(
        photo.name,
        f"[light_sea_green]{width} x {height} px[/light_sea_green]",
        f"{width / height:.3f}",
        common.colour_profile(photo) or "none",
    )
rprint(photo_table)


## Prompts

One prompt per photo, in the Star Colonel's two-part structure. The first part is his generic sentence verbatim. The second part describes only what that photo's own moment calls for: the child moves a little the way a real child would here, and every other person behaves naturally for the moment. Nothing else is added.

- **IMG_8840** - Henry holds a glass jar up to the camera with a broad grin; a girl at the right edge looks toward him and the jar. The second part uses the Star Colonel's own words for the girl
- **IMG_8564** - Henry poses for a birthday photo with a picture book, classmates around him
- **IMG_8732** - Henry pauses over a drawing with a blue pencil, absorbed, with no other person in the photo

The negative prompt is the LTX-2.5 reference default verbatim, followed by six terms against camera motion and redraw artefacts. The audio terms of the reference list are kept because the graph generates a sound track, even though the delivered clip carries only video.


In [ ]:
# The Star Colonel's generic sentence, verbatim
GENERIC_SENTENCE = "take the photo and make a small 5-10s animated clip out of this, essentially making the image live;"

# The photo-specific second part, one per working copy
SECOND_PARTS = {
    "IMG_8840": (
        "the boy holding the jar up to the camera keeps his grin and moves his head a little, the way a real "
        "child would; the girl on the right - let her not react to the thing held in the hands by the boy, "
        "she just looks for a short while and turns to another thing"
    ),
    "IMG_8564": (
        "the boy posing with his picture book grins and sways a little as he holds the pose, glancing briefly "
        "to his side and back to the camera, the way a real child would; the children around him behave "
        "naturally for the moment"
    ),
    "IMG_8732": (
        "the boy pauses over his drawing, his eyes moving between the paper and the blue pencil in his hand "
        "and his head following them a little, the way a real child would"
    ),
}

PROMPTS = {photo.stem: f"{GENERIC_SENTENCE} {SECOND_PARTS[photo.stem]}" for photo in WORKING_COPIES}

prompt_table = Table(title="Prompts", box=box.SIMPLE_HEAVY)
prompt_table.add_column("Photo", style="grey70")
prompt_table.add_column("Prompt", overflow="fold")
for stem, prompt in PROMPTS.items():
    prompt_table.add_row(stem, prompt)
rprint(
    Group(
        prompt_table,
        f"[bold slate_blue1]Negative prompt[/bold slate_blue1]\n  [dim]{NEGATIVE_PROMPT}[/dim]",
    )
)


## Rendering

Starts the model's ComfyUI server on the pinned GPU and renders one clip per working copy: 3 clips. For each clip the module copies the photo into the server's input folder, fills the graph markers (image, prompt, negative prompt, seed, width, height, length), submits, waits for the finished history entry, moves the native mp4 out, frees GPU room if the interpolation needs it, runs FILM to 30 fps, and moves every file the render added to ComfyUI's input, output and temp folders to the trash. A clip whose checkpoint records the same model, revision and settings is restored instead of rendered again; when a clip is rendered, its previous mp4 and sidecar go to the trash first, so the folder holds only the latest render. Time runs from submission to the finished history entry; peak VRAM is sampled with nvidia-smi during the render.


In [ ]:
server = i2v.Server(SETUP)

results, restored = [], set()
with Progress(
    SpinnerColumn(),
    TextColumn("[steel_blue]{task.description}"),
    BarColumn(bar_width=40),
    MofNCompleteColumn(),
    TimeElapsedColumn(),
    refresh_per_second=10,
) as progress:
    task = progress.add_task("Rendering clips", total=len(WORKING_COPIES))
    for photo in WORKING_COPIES:
        clip = OUTPUT_DIR / f"{photo.stem}.mp4"
        progress.update(task, description=f"Rendering {clip.name}")
        previous = common.load_result(clip)
        result = server.render(
            photo,
            clip,
            "default",
            {
                "{{PROMPT}}": PROMPTS[photo.stem],
                "{{NEGATIVE_PROMPT}}": NEGATIVE_PROMPT,
                "{{SEED}}": SEED,
                "{{WIDTH}}": WIDTH,
                "{{HEIGHT}}": HEIGHT,
                "{{LENGTH}}": LENGTH,
            },
            MODEL_SETTINGS,
        )
        if previous is not None and result == previous:
            restored.add(result.clip)
        results.append(result)
        progress.advance(task)
    # force completion (prevents N-1 issue)
    progress.update(task, completed=len(WORKING_COPIES), description="Rendered clips")
    progress.refresh()


## Save/Export

Each render already wrote its clip, sidecar and checkpoint. This cell lists the output folder, lists every file left in ComfyUI's input, output and temp folders to show that no copy of a photo or clip remains there, and stops the server.


In [ ]:
server.stop()
written = sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file())
comfyui_files = sorted(
    path for folder in i2v.WORK_FOLDERS for path in SETUP.io_dirs[folder].rglob("*") if path.is_file()
)
file_lines = "\n".join(
    f"  [cadet_blue]{path.relative_to(OUTPUT_DIR)}[/cadet_blue] [dim]{path.stat().st_size / 1e6:.2f} MB[/dim]"
    for path in written
)
comfyui_lines = "\n".join(f"  [cadet_blue]{path.name}[/cadet_blue] [dim]{path.parent}[/dim]" for path in comfyui_files)
stopped = not server.answers()
rprint(
    f"[bold medium_purple]Output folder[/bold medium_purple] [dim]{OUTPUT_DIR}[/dim]\n{file_lines}\n\n"
    f"[bold slate_blue1]Files in ComfyUI's {', '.join(i2v.WORK_FOLDERS)} folders[/bold slate_blue1] "
    f"[dim](ComfyUI's own files only)[/dim]\n"
    f"{comfyui_lines or '  [dim]none[/dim]'}\n\n"
    f"[bold slate_blue1]ComfyUI server[/bold slate_blue1]\n"
    f"  Stopped: [{'dark_sea_green' if stopped else 'indian_red'}]{'✓ yes' if stopped else '✗ no'}[/]"
)


## Results

Measurements per clip, and each clip under its source photo.

- **Time** - wall seconds from submitting the graph to the finished history entry: model load, 30 sampling steps, tiled VAE decode and the H.264 encode of the native clip; the FILM interpolation runs after that clock stops
- **Peak VRAM** - the highest nvidia-smi `memory.used` of the pinned GPU sampled during the render, minus the memory in use before the server started
- **Native fps** - the rate the graph rendered; the delivered clip is 30 fps
- **Interpolation** - FILM fp16 on the same GPU converts the 241 native frames to 300 at 30 fps; the method is recorded in the sidecar


In [ ]:
result_table = Table(title="LTX-2.5 clips", box=box.SIMPLE_HEAVY)
for column in ("Clip", "Seconds", "Peak VRAM", "Size", "Frames", "FPS", "Duration", "Native fps", "Measured"):
    result_table.add_column(column, justify="left" if column == "Clip" else "right")
for result in results:
    result_table.add_row(
        f"[grey70]{Path(result.clip).name}[/grey70]",
        f"[light_sea_green]{result.seconds:.1f} s[/light_sea_green]",
        f"[light_sea_green]{result.peak_vram_gb:.2f} GB[/light_sea_green]",
        f"{result.width} x {result.height}",
        f"[dark_sea_green]{result.frames}[/dark_sea_green]",
        f"[dark_sea_green]{result.fps:g}[/dark_sea_green]",
        f"[light_sea_green]{result.frames / result.fps:.2f} s[/light_sea_green]",
        f"{SETUP.native_fps:g}",
        "[dim]checkpoint[/dim]" if result.clip in restored else "this run",
    )
rprint(result_table)


In [ ]:
# Photo and clip linked by path relative to the notebook: no image or video bytes are stored here
for result in results:
    photo = INTERIM_DATA_DIR / PHOTO_SET / f"{Path(result.clip).stem}.png"
    display(Markdown(f"### {photo.stem}"))
    display(Image(url=os.path.relpath(photo), width=400))
    display(Markdown(f"`{Path(result.clip).name}`"))
    display(Video(os.path.relpath(PROJ_ROOT / result.clip), embed=False, width=400, html_attributes="controls loop muted"))


## Conclusions

LENGTH 241 is the single-pass probe, and at 10 s it settles. The notebook itself was last executed at 5 s, while the 241-frame probe runs are recorded in `logs/ltx-10s.log` (IMG_7666, 2026-10-25 22:12 to 22:34) and `logs/slowmo-probe-ltx60.log` (IMG_7666 at 60 steps).

The production method is the two-part chain at `tmp/workflows/ltx25-preview/render_ltx_chain_10s.py`: two 121-frame natives at 24 fps, part B starting from part A's last frame, joined to 242 frames and interpolated to 300 frames at 30 fps. That chain produced the eight clips and sidecars in `tmp/media/ltx25-i2v-10s/`.

- **This notebook** - LENGTH 241, one native pass at 24 fps (10.04 s), the settling probe
- **Delivered set** - `tmp/media/ltx25-i2v-10s/`, produced by the chain, not by a run of this notebook
- **Measurement** - how believable the motion looks is not measured here; the Star Colonel judges the clips
